# Sprint 5–6: Evaluation

Approach A (distribution matching) + Approach B (train-on-synthetic/test-on-real NCF).
Produces the final paper results table.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import yaml
import pandas as pd
import numpy as np

with open('../configs/mind_small.yaml') as f:
    config = yaml.safe_load(f)
ds = config['dataset']
sim_cfg = config['simulation']
eval_cfg = config['evaluation']

In [ ]:
# Load all artefacts
from data.mind_loader import load_news, load_behaviors, filter_active_users, temporal_split
from data.news_preprocessor import load_index
from simulation.db import fetch_all_interactions, fetch_all_users

news_df = load_news(ds['train_news_path'])
behaviors_df = load_behaviors(ds['dev_behaviors_path'])
behaviors_df = filter_active_users(behaviors_df, min_clicks=ds['min_real_user_clicks'])
_, real_test_df = temporal_split(behaviors_df, test_ratio=ds['test_ratio'])

real_train_df = load_behaviors(ds['train_behaviors_path'])

index, corpus_ids, embeddings, db_con = load_index(
    ds['index_path'], ds['ids_path'], ds['embeddings_path'], ds['db_path']
)
sim_interactions = fetch_all_interactions(sim_cfg['sim_db_path'])
sim_df = pd.DataFrame(sim_interactions)

print(f'Real test behaviors: {len(real_test_df):,} rows, {real_test_df["user_id"].nunique():,} users')
print(f'Simulation interactions: {len(sim_df):,} rows, {sim_df["user_id"].nunique():,} users')

## Approach A: Distribution Matching

In [ ]:
# Build virtual-user interest vectors from final simulation state
from recommender.profile_builder import build_profile
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(config['ir']['model'])

virtual_users = fetch_all_users(sim_cfg['sim_db_path'])
virtual_profiles = []
for user in virtual_users:
    user_interactions = [i for i in sim_interactions if i['user_id'] == user['user_id']]
    profile = build_profile(user_interactions, embed_model)
    if profile is not None:
        virtual_profiles.append(profile)
print(f'Built {len(virtual_profiles)} virtual user interest vectors')

In [ ]:
# Embeddings lookup function
id_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}
def embeddings_lookup(doc_id: str):
    idx = id_to_idx.get(str(doc_id))
    return embeddings[idx] if idx is not None else None

from evaluation.approach_a import run_approach_a

approach_a_results = run_approach_a(
    real_behaviors_df=real_test_df,
    sim_interactions_df=sim_df,
    news_df=news_df,
    virtual_user_profiles=virtual_profiles,
    embeddings_lookup=embeddings_lookup,
    k=eval_cfg['k'],
)

## Approach B: Train on Synthetic / Test on Real

In [ ]:
from evaluation.approach_b import run_approach_b

approach_b_results = run_approach_b(
    sim_interactions=sim_interactions,
    real_train_behaviors=real_train_df,
    real_test_behaviors=real_test_df,
    ncf_config=config['recommender'],
    ablation_sizes=eval_cfg['n_ablation_sizes'],
    k=eval_cfg['k'],
    seed=sim_cfg['seed'],
)

## Export Synthetic Dataset

In [ ]:
from export.trec_export import export_trec_qrels
from export.mind_export import export_behaviors_tsv

export_trec_qrels(sim_interactions, 'outputs/synthetic_qrels.txt')
export_behaviors_tsv(sim_interactions, 'outputs/synthetic_behaviors.tsv')
print('Exports complete.')

In [ ]:
# Optional: export as RecSim SequenceExamples
# from export.recsim_export import export_sequence_examples
# export_sequence_examples(sim_interactions, 'outputs/synthetic_trajectories.tfrecord')

## Summary Table (Paper-Ready)

In [ ]:
k = eval_cfg['k']
print('\n=== APPROACH A: Fidelity Metrics ===')
print(f"  CTR KL-divergence:          {approach_a_results.get('ctr_kl_divergence', 'N/A'):.4f}")
print(f"  Session-length Wasserstein: {approach_a_results.get('session_length_wasserstein', 'N/A'):.2f}")
print(f"  Category entropy gap:       {approach_a_results.get('category_entropy_gap', 'N/A'):.4f}")
print(f"  Replay NDCG@{k}:             {approach_a_results.get('replay_ndcg_at_k', 'N/A'):.4f}")

print('\n=== APPROACH B: NCF Results ===')
for row in approach_b_results.get('results', []):
    print(f"  {row['setting']:<35} NDCG@{k}={row.get(f'ndcg_at_{k}', 0.0):.4f}  Recall@{k}={row.get(f'recall_at_{k}', 0.0):.4f}")